In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

In [2]:
# Definimos los modelos y sus parrillas de parámetros
model_params = {
    'svm': {
        'model': SVC(probability=True),
        'params': {
            'C': [1, 10, 100],
            'kernel': ['rbf', 'linear']
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20]
        }
    },
    'xgboost': {
        'model': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.1, 0.01],
            'max_depth': [3, 6, 10]
        }
    }
}

In [5]:
from sklearn.preprocessing import LabelEncoder

# 1. Cargar el dataset híbrido (Reales + VAE)
df = pd.read_csv('embeddings_entrenamiento_vae.csv')

# 2. Preparar X e y
X = df.drop(['DIAG PSQ', 'Origen'], axis=1)
y = df['DIAG PSQ']

# 3. Codificar etiquetas de texto a números para todos los clasificadores
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('Clases originales:', le.classes_)
print('Clases codificadas:', np.unique(y_encoded))

# Usa y_encoded en fit() del GridSearchCV
y = y_encoded

Clases originales: ['F20' 'F21' 'F22' 'F23' 'F25' 'F29' 'F60.1']
Clases codificadas: [0 1 2 3 4 5 6]


In [6]:
scores = []

for model_name, mp in model_params.items():
    clf = GridSearchCV(mp['model'], mp['params'], cv=5, scoring='f1_macro', return_train_score=False)
    clf.fit(X, y)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:47:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:47:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:47:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:48:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "

In [7]:
# Convertir resultados a DataFrame para tu memoria
resultados_df = pd.DataFrame(scores, columns=['model', 'best_score', 'best_params'])
print(resultados_df)

           model  best_score  \
0            svm    0.560072   
1  random_forest    0.468491   
2        xgboost    0.517151   

                                         best_params  
0                       {'C': 1, 'kernel': 'linear'}  
1           {'max_depth': None, 'n_estimators': 100}  
2  {'learning_rate': 0.1, 'max_depth': 3, 'n_esti...  
